# Question 1: LeNet-5 Implementation

In [2]:
from datetime import datetime
print(datetime.now())
print("F22-3162")

import torch
import torch.nn as nn

class LeNet5(nn.Module):
    def __init__(self):
        super(LeNet5, self).__init__()
        # Conv1: 1 input channel (grayscale), 6 filters, 5x5 kernel
        self.conv1 = nn.Conv2d(1, 6, kernel_size=5)
        self.pool = nn.AvgPool2d(kernel_size=2, stride=2)
        # Conv2: 6 input channels, 16 filters, 5x5 kernel
        self.conv2 = nn.Conv2d(6, 16, kernel_size=5)
        # FC Layers: Flattened size is 16*5*5 = 400
        self.fc1 = nn.Linear(400, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        print(f"Input Shape: {x.shape}")
        x = self.pool(torch.tanh(self.conv1(x)))
        print(f"After Layer 1 (Conv+Pool): {x.shape}")
        
        x = self.pool(torch.tanh(self.conv2(x)))
        print(f"After Layer 2 (Conv+Pool): {x.shape}")
        
        x = x.view(-1, 400) 
        print(f"After Flatten: {x.shape}")
        
        x = torch.tanh(self.fc1(x))
        print(f"After FC1: {x.shape}")
        
        x = torch.tanh(self.fc2(x))
        print(f"After FC2: {x.shape}")
        
        x = self.fc3(x)
        print(f"Final Output: {x.shape}")
        return x

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

model = LeNet5()
dummy_batch = torch.randn(1, 1, 32, 32)
output = model(dummy_batch)

print(f"Total Trainable Parameters: {count_parameters(model)}")

2026-03-31 10:41:55.615813
F22-3162
Input Shape: torch.Size([1, 1, 32, 32])
After Layer 1 (Conv+Pool): torch.Size([1, 6, 14, 14])
After Layer 2 (Conv+Pool): torch.Size([1, 16, 5, 5])
After Flatten: torch.Size([1, 400])
After FC1: torch.Size([1, 120])
After FC2: torch.Size([1, 84])
Final Output: torch.Size([1, 10])
Total Trainable Parameters: 61706


# Question 1 Report
Mathematical Verification of Shapes:
* Input: 32x32
* Conv1 (5x5): $(32-5+1) = 28 \times 28$
* AvgPool1 (2x2): $28/2 = 14 \times 14$
* Conv2 (5x5): $(14-5+1) = 10 \times 10$
* AvgPool2 (2x2): $10/2 = 5 \times 5$
* Flatten: $16 \text{ filters} \times 5 \times 5 = 400$

In [4]:
from datetime import datetime
print(datetime.now())
print("F22-3162")

import torch
import torch.nn as nn

class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super(ResidualBlock, self).__init__()
        
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        self.shortcut = nn.Identity()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        identity = self.shortcut(x)
        
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        
        out = self.conv2(out)
        out = self.bn2(out)
        
        out += identity
        out = self.relu(out)
        
        print(f"Residual Block Output Shape: {out.shape}")
        return out

# Test Case 1:
print("Running Test Case 1")
block1 = ResidualBlock(64, 128, stride=2)
input1 = torch.randn(1, 64, 56, 56)
output1 = block1(input1)

# Test Case 2:
print("\nRunning Test Case 2")
block2 = ResidualBlock(128, 128, stride=1)
input2 = torch.randn(1, 128, 28, 28)
output2 = block2(input2)

2026-03-31 11:04:48.429553
F22-3162
Running Test Case 1
Residual Block Output Shape: torch.Size([1, 128, 28, 28])

Running Test Case 2
Residual Block Output Shape: torch.Size([1, 128, 28, 28])


## Report Question 2 
* This implementation handles the two main ways a residual block works: staying the same size or changing dimensions. In Test Case 1, the block sees that the input doesn't match the output and the stride is 2, so it uses a $1 \times 1$ convolution shortcut to shrink the image to $28 \times 28$ and double the channels. This matches output= 28. In Test Case 2, since the input and output are already identical $128 \times 28 \times 28$, it just uses a simple identity pass through. By adding Batch Normalization and ReLU after the convolutions, the model stays stable while learning the residual difference F(x) before adding it back to the original input x.